# OISST model comparison

Select a GPU runtime and Run all once. Code is installed from the latest main
of chahineNejm/Generalized-Patchwork; the resolved commit is saved.
Defaults live in experiments/default_hyperparameters.py. Edit overrides
below, not model definitions. No model or dataset implementation is embedded.
Then rerun only the stage you want. Failed stages block their dependents;
rerunning a failed stage is an explicit retry, never an automatic recovery.


## 1. Load code and dependencies


In [1]:
if '_OISST_STAGES' not in globals():
    _OISST_STAGES = {}
for _stage in ['checkout', 'install', 'settings', 'data', 'training', 'analysis', 'download']:
    _OISST_STAGES[_stage] = False
for _required in []:
    if not _OISST_STAGES.get(_required, False):
        raise RuntimeError(f'Run the {_required} stage successfully first')
import hashlib, importlib, json, os, subprocess, sys
from pathlib import Path
REPOSITORY = 'https://github.com/chahineNejm/Generalized-Patchwork.git'
BRANCH = 'main'
ROOT = Path('/content/gpw_oisst')
if ROOT.exists():
    raise FileExistsError('Checkout already exists; use a fresh Colab runtime')
if any((name == 'pwgp' or name.startswith('pwgp.') or name == 'experiments' or name.startswith('experiments.') for name in sys.modules)):
    raise RuntimeError('Restart the runtime before loading a new source version')
os.environ['GIT_TERMINAL_PROMPT'] = '0'
subprocess.run(['git', 'clone', '--filter=blob:none', '--no-checkout', '--branch', BRANCH, '--single-branch', REPOSITORY, str(ROOT)], check=True)
subprocess.run(['git', '-C', str(ROOT), 'sparse-checkout', 'set', '--no-cone', '/src/', '/experiments/', '/pyproject.toml', '/README.md'], check=True)
subprocess.run(['git', '-C', str(ROOT), 'checkout', BRANCH], check=True)
COMMIT = subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], text=True).strip()
SOURCE_ZIP = subprocess.check_output(['git', '-C', str(ROOT), 'archive', '--format=zip', COMMIT, 'pyproject.toml', 'README.md', 'src', 'experiments'])
SOURCE_PROVENANCE = {'repository': REPOSITORY, 'branch': BRANCH, 'commit': COMMIT, 'source_sha256': hashlib.sha256(SOURCE_ZIP).hexdigest()}
os.environ['MPLCONFIGDIR'] = str(ROOT / 'results/.matplotlib')
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.chdir(ROOT)
sys.path[:0] = [str(ROOT / 'src'), str(ROOT)]
importlib.invalidate_caches()
_GPW_CELL_RESULT = print(SOURCE_PROVENANCE)
_OISST_STAGES['checkout'] = True
_GPW_CELL_RESULT


{'repository': 'https://github.com/chahineNejm/Generalized-Patchwork.git', 'branch': 'main', 'commit': 'fe88227bde5a2ec9bd8b7cba0ccfae3d4dee2cc8', 'source_sha256': '51d7b10eaaeb66a665e713e600e2388cc32c1717ecea487d813045f9a9d83a4e'}


In [2]:
if '_OISST_STAGES' not in globals():
    _OISST_STAGES = {}
for _stage in ['install', 'settings', 'data', 'training', 'analysis', 'download']:
    _OISST_STAGES[_stage] = False
for _required in ['checkout']:
    if not _OISST_STAGES.get(_required, False):
        raise RuntimeError(f'Run the {_required} stage successfully first')
_GPW_CELL_RESULT = subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--editable', str(ROOT) + '[data,ukhousing]'], check=True)
_OISST_STAGES['install'] = True
_GPW_CELL_RESULT


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '--quiet', '--editable', '/content/gpw_oisst[data,ukhousing]'], returncode=0)

## 2. Defaults and explicit overrides


In [3]:
if '_OISST_STAGES' not in globals():
    _OISST_STAGES = {}
for _stage in ['settings', 'training', 'analysis', 'download']:
    _OISST_STAGES[_stage] = False
for _required in ['install']:
    if not _OISST_STAGES.get(_required, False):
        raise RuntimeError(f'Run the {_required} stage successfully first')
from dataclasses import replace
from datetime import UTC, datetime
from IPython.display import Image, display
from google.colab import files
import pandas as pd
import torch
from experiments.default_hyperparameters import oisst_defaults
from experiments.oisst_benchmark import archive_directory, completed_run_directories, load_prepared, prepare_experiment, run_prepared_comparison
from experiments.run_selection import select_runs
DATA, RUNS = oisst_defaults(device='cuda:0')
MODELS = ['SVGP', 'GPW', 'ProSpar', 'VNNGP', 'Standard Patchwork']
#OVERRIDES = {'SVGP': {'epochs': 2, 'warmup_epochs': 1}, 'GPW': {'epochs': 2, 'warmup_epochs': 1}, 'ProSpar': {'epochs': 2, 'warmup_epochs': 1}, 'VNNGP': {'epochs': 2, 'warmup_epochs': 1}, 'Standard Patchwork': {'epochs': 2}}
OVERRIDES = {}
DATA_OVERRIDES = {}
RUNS = select_runs(RUNS, models=MODELS, overrides=OVERRIDES)
DATA = replace(DATA, **DATA_OVERRIDES)
WORK_ROOT = ROOT / 'results/oisst'
display(pd.DataFrame([{'run': name, **config.__dict__} for name, config in RUNS]))
_GPW_CELL_RESULT = print(DATA)
_OISST_STAGES['settings'] = True
_GPW_CELL_RESULT


/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


NameError: name 'OVERRIDES' is not defined

## 3. Prepare once or explicitly load saved prepared data

Keep this stage unchanged when only MODELS or model hyperparameters change.
To restore saved data, select DATA_ACTION="load" and set PREPARED_PATH.
No missing or mismatched data are rebuilt automatically.


In [ ]:
if '_OISST_STAGES' not in globals():
    _OISST_STAGES = {}
for _stage in ['data', 'training', 'analysis', 'download']:
    _OISST_STAGES[_stage] = False
for _required in ['install', 'settings']:
    if not _OISST_STAGES.get(_required, False):
        raise RuntimeError(f'Run the {_required} stage successfully first')
DATA_ACTION = 'prepare'
PREPARED_PATH = None
if DATA_ACTION == 'prepare':
    PREPARED_DIRECTORY = WORK_ROOT / datetime.now(UTC).strftime('prepared_%Y%m%dT%H%M%S%fZ')
    prepare_experiment(DATA, directory=PREPARED_DIRECTORY, data_root=ROOT / 'data', source_provenance=SOURCE_PROVENANCE, source_zip=SOURCE_ZIP)
elif DATA_ACTION == 'load':
    if PREPARED_PATH is None:
        raise ValueError('Set PREPARED_PATH explicitly before loading prepared data')
    PREPARED_DIRECTORY = Path(PREPARED_PATH)
    load_prepared(PREPARED_DIRECTORY, data_config=DATA)
else:
    raise ValueError('DATA_ACTION must be prepare or load')
print('Prepared data:', PREPARED_DIRECTORY)
_GPW_CELL_RESULT = display(Image(filename=str(PREPARED_DIRECTORY / 'exploratory/targets.png')))
_OISST_STAGES['data'] = True
_GPW_CELL_RESULT


## 4. Train only the selected models

Edit MODELS/OVERRIDES and rerun settings, then this cell. Prepared data
are reused only after config/checksum validation. Each execution creates
a new batch and trains selected models from scratch; previous results stay.
After failure, select only the model you wish to retry and rerun this cell.
No automatic checkpoint resumption, skipping or overwriting.


In [ ]:
if '_OISST_STAGES' not in globals():
    _OISST_STAGES = {}
for _stage in ['training', 'analysis', 'download']:
    _OISST_STAGES[_stage] = False
for _required in ['install', 'settings', 'data']:
    if not _OISST_STAGES.get(_required, False):
        raise RuntimeError(f'Run the {_required} stage successfully first')
live = display('Starting OISST comparison...', display_id=True)

def show_progress(path, epoch):
    live.update(Image(filename=str(path / 'visuals/progress.png')))
    display(Image(filename=str(path / 'visuals/training.png')))

def show_completed(path, archive, metrics):
    display(pd.DataFrame([metrics]).T)
    display(Image(filename=str(path / 'visuals/maps.png')))
    display(Image(filename=str(path / 'visuals/metrics.png')))
    display(Image(filename=str(path / 'visuals/hyperparameters.png')))
    metadata = json.loads((path / 'model_metadata.json').read_text())
    if metadata['inducing_positions_saved']:
        display(Image(filename=str(path / 'visuals/inducing_positions.png')))
    files.download(str(archive))
BATCH_DIRECTORY = WORK_ROOT / datetime.now(UTC).strftime('batch_%Y%m%dT%H%M%S%fZ')
result = run_prepared_comparison(DATA, RUNS, prepared_directory=PREPARED_DIRECTORY, directory=BATCH_DIRECTORY, source_provenance=SOURCE_PROVENANCE, source_zip=SOURCE_ZIP, on_snapshot=show_progress, on_complete=show_completed)
display(pd.read_json(result / 'comparison.json'))
_GPW_CELL_RESULT = print('Results:', result)
_OISST_STAGES['training'] = True
_GPW_CELL_RESULT


## 5. Recompute metrics and figures without training

Select latest_batch for the last successful training batch, or saved_runs
and fill SAVED_RUNS with exact completed model-run directories. This stage
reads saved predictions only. Original results are never overwritten.
The saved_runs action can be used after setup without data preparation or
training, including to inspect earlier completed runs after another fails.


In [ ]:
if '_OISST_STAGES' not in globals():
    _OISST_STAGES = {}
for _stage in ['analysis', 'download']:
    _OISST_STAGES[_stage] = False
for _required in ['install']:
    if not _OISST_STAGES.get(_required, False):
        raise RuntimeError(f'Run the {_required} stage successfully first')
from datetime import UTC, datetime
from experiments.exploratory_analysis import recompute_saved_runs
from experiments.oisst_benchmark import completed_run_directories
from IPython.display import Image, display
import pandas as pd
ANALYSIS_SOURCE = 'latest_batch'
SAVED_RUNS = []
if ANALYSIS_SOURCE == 'latest_batch':
    if not _OISST_STAGES.get('training', False):
        raise RuntimeError('Latest training stage failed or has not run; explicitly select saved_runs instead')
    analysis_runs = completed_run_directories(BATCH_DIRECTORY)
elif ANALYSIS_SOURCE == 'saved_runs':
    analysis_runs = SAVED_RUNS
else:
    raise ValueError('ANALYSIS_SOURCE must be latest_batch or saved_runs')
ANALYSIS_DIRECTORY = ROOT / 'results/oisst' / datetime.now(UTC).strftime('analysis_%Y%m%dT%H%M%S%fZ')
recompute_saved_runs(analysis_runs, destination=ANALYSIS_DIRECTORY)
analysis_table = pd.read_json(ANALYSIS_DIRECTORY / 'comparison.json')
display(analysis_table)
for path in analysis_table['analysis_directory']:
    display(Image(filename=str(Path(path) / 'visuals/metrics.png')))
    display(Image(filename=str(Path(path) / 'visuals/maps.png')))
_GPW_CELL_RESULT = print('Analysis:', ANALYSIS_DIRECTORY)
_OISST_STAGES['analysis'] = True
_GPW_CELL_RESULT


## 6. Download a batch, analysis or explicitly named directory


In [ ]:
if '_OISST_STAGES' not in globals():
    _OISST_STAGES = {}
for _stage in ['download']:
    _OISST_STAGES[_stage] = False
for _required in ['install']:
    if not _OISST_STAGES.get(_required, False):
        raise RuntimeError(f'Run the {_required} stage successfully first')
from datetime import UTC, datetime
from experiments.oisst_benchmark import archive_directory
from google.colab import files
DOWNLOAD_SOURCE = 'latest_batch'
SAVED_DIRECTORY = None
if DOWNLOAD_SOURCE == 'latest_batch':
    if not _OISST_STAGES.get('training', False) or not _OISST_STAGES.get('analysis', False):
        raise RuntimeError('Complete the latest training/analysis stages or explicitly select saved_directory')
    download_directory = BATCH_DIRECTORY
elif DOWNLOAD_SOURCE == 'analysis':
    if not _OISST_STAGES.get('analysis', False):
        raise RuntimeError('Complete the analysis stage first')
    download_directory = ANALYSIS_DIRECTORY
elif DOWNLOAD_SOURCE == 'saved_directory':
    if SAVED_DIRECTORY is None:
        raise ValueError('Set SAVED_DIRECTORY explicitly')
    download_directory = Path(SAVED_DIRECTORY)
else:
    raise ValueError('Unknown DOWNLOAD_SOURCE')
archive = archive_directory(download_directory, destination=ROOT / 'results/oisst' / datetime.now(UTC).strftime('download_%Y%m%dT%H%M%S%fZ.zip'))
_GPW_CELL_RESULT = files.download(str(archive))
_OISST_STAGES['download'] = True
_GPW_CELL_RESULT
